In [1]:
import cv2
import ultralytics
from ultralytics import YOLO
import cvzone
import math

In [2]:
from sort import * 

In [14]:
cap=cv2.VideoCapture("trafficVideo2.mp4")
model=YOLO("../yoloModel/yolo26n.pt")


#mask=cv2.imread("mask.png")


tracker =Sort(max_age=20,min_hits=3,iou_threshold=0.3)

limit=[320,100,635,110]

totalCount=[]

ClassName=list(model.names.values())
while True:
    success,img=cap.read()
    #imgRegion =cv2.bitwise_and(img,mask)
    result = model(img, stream=True)
    detection=np.empty((0,5))
    for r in result:
        boxes = r.boxes
        for box in boxes:
            # Safely unpack tensor to Python list
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
            
            # Calculate width and height for cvzone
            w, h = x2 - x1, y2 - y1
            
            # Draw the bounding box corners
            cvzone.cornerRect(img, bbox=(x1, y1, w, h), l=4, rt=2)

            #Confidence of match
            conf = math.ceil((box.conf[0]*100))/100
            
            #Class Namec
            cls = int(box.cls[0])

            currentclass = ClassName[cls]
            if currentclass == 'car' or currentclass == 'bus' or currentclass == 'truck' or currentclass == 'motorbike':
                #cvzone.putTextRect(img,f" {currentclass} {conf}",(max(0,x1),max(15,y1)),scale=1.2,thickness=1)
                pass 
            CurrentArray=np.array([x1,y1,x2,y2,conf])
            detection = np.vstack((detection,CurrentArray))
    


    TrackerResult=tracker.update(detection)
    cv2.line(img,(limit[0],limit[1]),(limit[2],limit[3]),(0,0,0),2)
    for result in TrackerResult:
            x1,y1,x2,y2,Id=result
            x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
            w,h=x2-x1,y2-y1
            #cvzone.cornerRect(img,(x1,y1,w,h),l=9,rt=2,colorR=(255,192,0))
            cvzone.putTextRect(img,f"{int(Id)}",(max(0,x1),max(15,y1)),scale=1,thickness=1,offset=3)
            print(result)

            cx,cy=x1+w//2,y1+h//2
            cv2.circle(img,(cx,cy),2,(255,0,0),cv2.FILLED )
            
            if limit[0]<cx<limit[2] and limit[1]-10<cy<limit[1]+10:
                if totalCount.count(Id)==0:
                    totalCount.append(Id)
                    cv2.line(img,(limit[0],limit[1]),(limit[2],limit[3]),(0,255,0),2)
            cvzone.putTextRect(img,f"{len(totalCount)}",(50,50))







    cv2.imshow("window",img)
    #cv2.imshow("imgRegion",imgRegion)
    if cv2.waitKey(1) & 0xff==ord('x'):
       break

cap.release()
cv2.destroyAllWindows()


0: 384x640 1 person, 2 cars, 177.0ms
Speed: 4.1ms preprocess, 177.0ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)
[        114          37         146         124         177]
[        207          77         321         182         176]
[        124         113         262         238         175]

0: 384x640 1 person, 2 cars, 267.1ms
Speed: 11.5ms preprocess, 267.1ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)
[        114          37         146         124         177]
[     206.88       77.11      320.12      181.89         176]
[        124         113         262         238         175]

0: 384x640 1 person, 2 cars, 166.1ms
Speed: 2.9ms preprocess, 166.1ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640)
[     114.94          37      146.94         124         177]
[     207.23      77.778      320.66      182.16         176]
[        124         113         262         238         175]

0: 384x640 1 person, 2 cars, 194

In [11]:
print(model.names)

{0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 4: 'airplane', 5: 'bus', 6: 'train', 7: 'truck', 8: 'boat', 9: 'traffic light', 10: 'fire hydrant', 11: 'stop sign', 12: 'parking meter', 13: 'bench', 14: 'bird', 15: 'cat', 16: 'dog', 17: 'horse', 18: 'sheep', 19: 'cow', 20: 'elephant', 21: 'bear', 22: 'zebra', 23: 'giraffe', 24: 'backpack', 25: 'umbrella', 26: 'handbag', 27: 'tie', 28: 'suitcase', 29: 'frisbee', 30: 'skis', 31: 'snowboard', 32: 'sports ball', 33: 'kite', 34: 'baseball bat', 35: 'baseball glove', 36: 'skateboard', 37: 'surfboard', 38: 'tennis racket', 39: 'bottle', 40: 'wine glass', 41: 'cup', 42: 'fork', 43: 'knife', 44: 'spoon', 45: 'bowl', 46: 'banana', 47: 'apple', 48: 'sandwich', 49: 'orange', 50: 'broccoli', 51: 'carrot', 52: 'hot dog', 53: 'pizza', 54: 'donut', 55: 'cake', 56: 'chair', 57: 'couch', 58: 'potted plant', 59: 'bed', 60: 'dining table', 61: 'toilet', 62: 'tv', 63: 'laptop', 64: 'mouse', 65: 'remote', 66: 'keyboard', 67: 'cell phone', 68: 'microw